# m14 lab · Biopotential front ends

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Between the electrode and the number on the screen sit a differential amplifier, a reference, a driven right leg and an analog-to-digital converter. You compute how resistor tolerance and electrode impedance mismatch limit common-mode rejection, how a driven right leg shrinks the common mode, where the noise comes from, how electrode impedances shift the Wilson central terminal, and what oversampling buys.

| Part | What you do | Equation |
| --- | --- | --- |
| A | The difference amplifier and its CMRR | eq-m14-cmrr, eq-m14-diffamp |
| B | The instrumentation amplifier | eq-m14-inamp |
| C | Mains interference: electrode mismatch and the driven right leg | eq-m14-imbalance, eq-m14-drl |
| D | A noise budget | eq-m14-noise |
| E | The Wilson central terminal and the converter | eq-m14-wct, eq-m14-oversampling |

SI units unless stated. Component values, impedances and noise figures are illustrative choices, not the values of any product. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import frontend as F

## Part A · The difference amplifier and its CMRR

A four-resistor difference amplifier outputs $(R_2/R_1)(v_+ - v_-)$ when the resistors match. Mismatch lets some of the
common mode through. With tolerance $t$, the worst-case CMRR is $(1 + G)/(4t)$ (eq-m14-diffamp). Draw many random
amplifiers and compare.

In [ ]:
rng = np.random.default_rng(14)
G = 10.0
fig, ax = plt.subplots()
for tol in (0.01, 0.001):
    c = F.random_cmrr(G, tol, 20000, rng)
    ax.hist(F.to_db(c), bins=80, alpha=0.6, label=f"tolerance {tol:.1%}")
    ax.axvline(F.to_db(F.worst_case_cmrr(G, tol)), ls="--", c="k")
    print(f"tolerance {tol:.1%}: worst case {F.to_db(F.worst_case_cmrr(G, tol)):.1f} dB, "
          f"lowest drawn {F.to_db(c.min()):.1f} dB, median {F.to_db(np.median(c)):.1f} dB")
ax.set_xlabel("CMRR (dB)"); ax.set_ylabel("count"); ax.set_xlim(40, 130); ax.legend(); plt.show()

In [ ]:
# Checks
c = F.random_cmrr(G, 0.01, 20000, rng)
assert c.min() > 0.95 * F.worst_case_cmrr(G, 0.01)
assert abs(F.to_db(F.worst_case_cmrr(G, 0.001)) - F.to_db(F.worst_case_cmrr(G, 0.01)) - 20) < 1e-9

## Part B · The instrumentation amplifier

An input stage of two op amps multiplies the difference by $G_1 = 1 + 2R/R_G$ and passes the common mode at unity, so the
difference stage that follows sees a $G_1$-times better ratio (eq-m14-inamp). Use a worst-case 0.1% difference stage.

In [ ]:
t = 0.001
rs = (1 + t, 10 * (1 - t), 1 - t, 10 * (1 + t))
base = abs(np.divide(*F.diff_amp_gains(*rs)))
for rg in (1e12, 20e3, 2e3, 200.0):
    g1 = 1 + 2 * 10e3 / rg
    print(f"G1 = {g1:7.1f}: CMRR {F.to_db(F.inamp_cmrr(10e3, rg, *rs)):6.1f} dB "
          f"(difference stage alone {F.to_db(base):.1f} dB + 20 log10 G1 = {F.to_db(base * g1):.1f})")

In [ ]:
# Checks
assert abs(F.inamp_cmrr(10e3, 200.0, *rs) / (101 * base) - 1) < 1e-3

## Part C · Mains interference: electrode mismatch and the driven right leg

The body floats at a common-mode voltage driven by stray coupling to the mains. Unequal electrode impedances $Z_1$,
$Z_2$ into the amplifier's common-mode input impedance $Z_{cm}$ turn part of it into a difference, about
$v_{cm}\,\Delta Z/Z_{cm}$ (eq-m14-imbalance). A driven right leg feeds back the inverted common mode and divides it by
$1 + G$ (eq-m14-drl). Add both to a synthetic electrogram. The 1 mV electrogram and 10 mV common mode are
illustrative.

In [ ]:
from epml import recording as REC
fs = 2000.0
t_ms = np.arange(0, 400, 1000 / fs)
egm = REC.unipolar_egm(t_ms - 200.0)                       # m04 model, relative units
egm_mv = egm / np.max(np.abs(egm))                          # scale to 1 mV peak (illustrative)
vcm_mv = 10.0 * np.sin(2 * np.pi * 50.0 * t_ms / 1000)      # 10 mV, 50 Hz common mode (illustrative)
zcm = 1 / (2j * np.pi * 50.0 * 20e-12)                      # 20 pF input capacitance to ground (illustrative)
amp = 10 * F.worst_case_cmrr(10.0, 0.001)
fig, ax = plt.subplots()
for dz, G, label in [(2e3, 0, "2 kΩ mismatch"), (20e3, 0, "20 kΩ mismatch"), (20e3, 100, "20 kΩ, driven right leg, G = 100")]:
    leak = 1 / F.combined_cmrr(amp, F.imbalance_cmrr(dz, zcm))
    rec = egm_mv + leak * vcm_mv / (1 + G)
    ax.plot(t_ms, rec, label=label)
    print(f"{label:34s}: mains in the record {1000 * leak * 10 / (1 + G):8.3f} µV peak")
ax.set_xlabel("time (ms)"); ax.set_ylabel("recorded (mV)"); ax.legend(); plt.show()

In [ ]:
# Checks
assert abs(F.imbalance_cmrr(20e3, zcm) / F.imbalance_cmrr(2e3, zcm) - 0.1) < 1e-12
assert abs(F.body_common_mode(1.0, 1.0, 100) - 1 / 101) < 1e-15

## Part D · A noise budget

Three sources add in quadrature at the input: the source's thermal noise $\sqrt{4kTR_s}$ (Nyquist), the amplifier's
voltage noise $e_n$, and its current noise $i_n$ through $R_s$ (eq-m14-noise). The noise factor is lowest at
$R_s = e_n/i_n$. Halving an electrode's radius doubles its spreading resistance (m13).

In [ ]:
e_n, i_n, bw = 10e-9, 1e-13, 500.0            # illustrative amplifier and bandwidth
r = np.logspace(2, 7, 300)
plt.loglog(r, F.thermal_density(r) * 1e9, label="thermal")
plt.loglog(r, np.full_like(r, e_n * 1e9), label="amplifier voltage")
plt.loglog(r, i_n * r * 1e9, label="current × source")
plt.loglog(r, F.total_density(r, e_n, i_n) * 1e9, "k", lw=2, label="total")
plt.xlabel("source resistance (Ω)"); plt.ylabel("noise density (nV/√Hz)"); plt.legend(); plt.show()
for rs in (1e3, 2e3, 1e5):
    print(f"R_s = {rs:8.0f} Ω: {F.rms_noise(F.total_density(rs, e_n, i_n), bw) * 1e6:.3f} µV rms over {bw:.0f} Hz")
print("lowest noise factor at", F.optimal_source_resistance(e_n, i_n), "Ω")

In [ ]:
# Checks
assert abs(F.thermal_density(2e3) / F.thermal_density(1e3) - np.sqrt(2)) < 1e-12
k = np.argmin(F.noise_factor(r, e_n, i_n))
assert abs(np.log10(r[k] / F.optimal_source_resistance(e_n, i_n))) < 0.02

## Part E · The Wilson central terminal and the converter

The central terminal joins the three limb electrodes through resistors $R_W$. With electrode impedances $Z_i$ it is a
weighted mean, $\sum \Phi_i/(R_W + Z_i) / \sum 1/(R_W + Z_i)$ (eq-m14-wct), not the plain mean. Then the converter: an
$N$-bit converter after gain $G$ resolves full scale$/(G\,2^N)$, and averaging $m$ samples of white noise gains
$10\log_{10} m$ dB (eq-m14-oversampling).

In [ ]:
pots = np.array([0.4, 0.9, -0.3])                       # limb potentials, relative (illustrative)
for z_bad in (1e3, 10e3, 50e3):
    for rw in (5e3, 100e3, 10e6):
        print(f"one electrode {z_bad/1e3:4.0f} kΩ, R_W {rw/1e3:7.0f} kΩ: WCT error {F.wct(pots, [1e3, z_bad, 1e3], rw) - pots.mean():+.4f}")
print()
print("bits to span 200 mV with 1 µV steps:", F.bits_needed(0.2, 1e-6))
rng = np.random.default_rng(1)
x = rng.standard_normal(2 ** 18)
for m in (4, 16, 64):
    print(f"average {m:2d} samples: measured gain {20*np.log10(x.std() / F.average_blocks(x, m).std()):5.2f} dB, "
          f"theory {F.oversampling_gain_db(m):5.2f} dB")

In [ ]:
# Checks
assert abs(F.wct(pots, [1e3, 1e3, 1e3], 5e3) - pots.mean()) < 1e-12
assert abs(F.wct(pots, [1e3, 50e3, 1e3], 5e3) - pots.mean()) > 10 * abs(F.wct(pots, [1e3, 50e3, 1e3], 10e6) - pots.mean())
assert abs(20*np.log10(x.std() / F.average_blocks(x, 16).std()) - F.oversampling_gain_db(16)) < 0.2

## Where to go next

- Make the input capacitances unequal too, and see the imbalance error turn into a phase-shifted mains signal.
- Add a high-pass input coupling network (m04) and check how its resistor mismatch converts common mode at low
  frequency.
- m15 covers the patient-safety side of the same front end: isolation, leakage current and defibrillation protection.